## 0. Configuration

In [ ]:
import torch 

DATA_DIR    = '/content/data/'
GRAPH_DIR   = '/content/data/graphs/'
CKPT_DIR    = '/content/drive/MyDrive/jet_tagging/checkpoints/'

N_SAMPLES   = None       # None = use full split; set an int (e.g. 100_000) to subset
K_NEIGHBORS = 7
BATCH_SIZE  = 256
LR_BASE     = 1e-3 
N_EPOCHS    = 10

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")


## 1. Setup

In [ ]:
!pip install -q zenodo_get torch_geometric


In [ ]:
import os
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, GINConv, EdgeConv, global_mean_pool

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, accuracy_score

from tqdm.auto import tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(GRAPH_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)


## 2. Dataset download

**Top Quark Tagging Reference Dataset** (Kasieczka et al. 2019, [Zenodo doi:10.5281/zenodo.2603256](https://doi.org/10.5281/zenodo.2603256)).

- Three splits: `train` (1.2M jets), `val` (403K jets), `test` (404K jets).
- Each split is an HDF5 table with columns `E_i`, `PX_i`, `PY_i`, `PZ_i` for `i = 0..199`
  (up to 200 zero-padded constituent four-vectors per jet) plus the label column
  `is_signal_new` (1 = top jet, 0 = QCD background).
- The dataset is balanced: signal fraction ≈ 0.50 in every split.


In [ ]:
if not all(os.path.exists(os.path.join(DATA_DIR, f'{split}.h5')) for split in ['train', 'val', 'test']):
    !zenodo_get 2603256 -o {DATA_DIR}
else:
    print("Dataset already downloaded, skipping.")


## 3. HDF5 => numpy conversion

In [ ]:
FEAT_COLS = [f'{q}_{i}' for i in range(200) for q in ['E', 'PX', 'PY', 'PZ']]

def convert_to_numpy(hdf_path, out_prefix, chunk_size=50_000):
    store = pd.HDFStore(hdf_path, mode='r')
    total = store.get_storer('table').nrows
    store.close()

    X_chunks, y_chunks = [], []
    for start in range(0, total, chunk_size):
        df = pd.read_hdf(hdf_path, key='table', start=start, stop=start + chunk_size)
        X_chunks.append(df[FEAT_COLS].values.reshape(-1, 200, 4).astype(np.float32))
        y_chunks.append(df['is_signal_new'].values.astype(np.int64))
        del df

    np.save(out_prefix + '_X.npy', np.concatenate(X_chunks))
    np.save(out_prefix + '_y.npy', np.concatenate(y_chunks))


In [ ]:
for split in ['train', 'val', 'test']:
    x_path = os.path.join(DATA_DIR, f'{split}_X.npy')
    y_path = os.path.join(DATA_DIR, f'{split}_y.npy')
    if os.path.exists(x_path) and os.path.exists(y_path):
        print(f"{split}: already converted, skipping.")
        continue
    print(f"{split}: converting...")
    convert_to_numpy(os.path.join(DATA_DIR, f'{split}.h5'), os.path.join(DATA_DIR, split))
    print(f"{split}: done.")


## 4. Graph pre-computation

In [ ]:
def build_graph(row, k=K_NEIGHBORS):
    """row: (200, 4) array with columns (E, PX, PY, PZ) => PyG Data(x, edge_index)."""
    mask = row[:, 0] > 0                      # drop zero-padded particles
    particles = row[mask]
    E, PX, PY, PZ = particles[:, 0], particles[:, 1], particles[:, 2], particles[:, 3]

    pT  = np.sqrt(PX**2 + PY**2)
    p   = np.sqrt(PX**2 + PY**2 + PZ**2)
    eta = np.arctanh(np.clip(PZ / (p + 1e-8), -1 + 1e-7, 1 - 1e-7))
    phi = np.arctan2(PY, PX)

    pT_sum = pT.sum() + 1e-8
    E_sum  = E.sum() + 1e-8
    eta_jet = (pT * eta).sum() / pT_sum       # pT-weighted jet axis
    phi_jet = (pT * phi).sum() / pT_sum

    pT_rel    = pT / pT_sum
    delta_eta = eta - eta_jet
    delta_phi = (phi - phi_jet + np.pi) % (2 * np.pi) - np.pi
    E_rel     = E / E_sum

    x = np.stack([pT_rel, delta_eta, delta_phi, E_rel], axis=1)

    pos = np.stack([delta_eta, delta_phi], axis=1)
    k_actual = min(k, len(pos) - 1)
    nbrs = NearestNeighbors(n_neighbors=k_actual + 1).fit(pos)
    _, indices = nbrs.kneighbors(pos)
    src = np.repeat(np.arange(len(pos)), k_actual)
    dst = indices[:, 1:].flatten()            # drop self-loop (column 0)
    edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)

    return Data(x=torch.tensor(x, dtype=torch.float), edge_index=edge_index)


In [ ]:
# Sanity check
X_sample = np.load(os.path.join(DATA_DIR, 'train_X.npy'), mmap_mode='r')
g = build_graph(X_sample[0])
print("Nodes:", g.num_nodes)
print("Edges:", g.num_edges)
print("Node feature shape:", tuple(g.x.shape))


In [ ]:
def precompute_graphs(split, n_samples=None):
    X = np.load(os.path.join(DATA_DIR, f'{split}_X.npy'), mmap_mode='r')
    y = np.load(os.path.join(DATA_DIR, f'{split}_y.npy'))
    n = n_samples or len(X)

    out_dir = os.path.join(GRAPH_DIR, split)
    os.makedirs(out_dir, exist_ok=True)

    n_existing = len([f for f in os.listdir(out_dir) if f.endswith('.pt')])
    if n_existing >= n:
        print(f"{split}: {n_existing} graphs already computed, skipping.")
        return

    for i in tqdm(range(n), desc=f'Building {split} graphs'):
        out_path = os.path.join(out_dir, f'{i}.pt')
        if os.path.exists(out_path):
            continue
        graph = build_graph(X[i])
        graph.y = torch.tensor([y[i]], dtype=torch.long)
        torch.save(graph, out_path)


In [ ]:
for split in ['train', 'val', 'test']:
    precompute_graphs(split, n_samples=N_SAMPLES)


## 5. Dataset and DataLoader

In [ ]:
class JetDataset(Dataset):
    def __init__(self, graph_dir, n=None):
        super().__init__()
        self.graph_dir = graph_dir
        n_available = len([f for f in os.listdir(graph_dir) if f.endswith('.pt')])
        self.n = n or n_available

    def len(self):
        return self.n

    def get(self, idx):
        return torch.load(os.path.join(self.graph_dir, f'{idx}.pt'), weights_only=False)


In [ ]:
train_ds = JetDataset(os.path.join(GRAPH_DIR, 'train'), n=N_SAMPLES)
val_ds   = JetDataset(os.path.join(GRAPH_DIR, 'val'),   n=N_SAMPLES)
test_ds  = JetDataset(os.path.join(GRAPH_DIR, 'test'),  n=N_SAMPLES)

pin_memory = DEVICE == 'cuda'
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=pin_memory)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin_memory)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin_memory)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


## 6. Model architectures

In [ ]:
def classifier_head():
    return nn.Sequential(
        nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 2)
    )


class JetGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(4, 64)
        self.conv2 = GCNConv(64, 64)
        self.classifier = classifier_head()

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)


In [ ]:
class JetGIN(nn.Module):
    def __init__(self):
        super().__init__()
        def mlp(in_ch, out_ch):
            return nn.Sequential(
                nn.Linear(in_ch, out_ch), nn.BatchNorm1d(out_ch), nn.ReLU(),
                nn.Linear(out_ch, out_ch),
            )
        self.conv1 = GINConv(mlp(4, 64))
        self.conv2 = GINConv(mlp(64, 64))
        self.classifier = classifier_head()

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)


In [ ]:
class JetGAT(nn.Module):
    def __init__(self, heads=4):
        super().__init__()
        self.conv1 = GATConv(4,  16, heads=heads, concat=True)   # -> 64
        self.conv2 = GATConv(64, 64, heads=heads, concat=False)   # -> 64
        self.classifier = classifier_head()

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)


class JetEdgeConv(nn.Module):
    def __init__(self):
        super().__init__()
        def edge_mlp(in_ch, out_ch):
            return nn.Sequential(
                nn.Linear(in_ch * 2, out_ch), nn.BatchNorm1d(out_ch), nn.ReLU(),
                nn.Linear(out_ch,    out_ch), nn.BatchNorm1d(out_ch), nn.ReLU(),
            )
        self.conv1 = EdgeConv(edge_mlp(4,  64), aggr='max')
        self.conv2 = EdgeConv(edge_mlp(64, 64), aggr='max')
        self.classifier = classifier_head()

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.classifier(x)


In [ ]:
MODEL_CLASSES = {
    'GCN':      JetGCN,
    'GIN':      JetGIN,
    'GAT':      JetGAT,
    'EdgeConv': JetEdgeConv,
}

print(f"{'Model':<10} {'Params':>10}")
print("-" * 22)
for name, cls in MODEL_CLASSES.items():
    n_params = sum(p.numel() for p in cls().parameters())
    print(f"{name:<10} {n_params:>10,}")


## 7. Trainer

In [ ]:
def _bg_rejection(labels, probs, signal_eff):
    threshold = np.percentile(probs[labels == 1], (1 - signal_eff) * 100)
    bg_eff = (probs[labels == 0] >= threshold).mean()
    return 1.0 / bg_eff if bg_eff > 0 else float('inf')


class Trainer:
    def __init__(self, model, optimizer, scheduler=None, device='cpu'):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.criterion = nn.CrossEntropyLoss()
        self.best_val_loss = float('inf')
        self.history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

    def _run_epoch(self, loader, train):
        self.model.train(train)
        total_loss, correct, total = 0.0, 0, 0
        for batch in loader:
            batch = batch.to(self.device)
            if train:
                self.optimizer.zero_grad()
            with torch.set_grad_enabled(train):
                out = self.model(batch.x, batch.edge_index, batch.batch)
                loss = self.criterion(out, batch.y.squeeze())
            if train:
                loss.backward()
                self.optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            correct += (out.argmax(1) == batch.y.squeeze()).sum().item()
            total += batch.num_graphs
        return total_loss / total, correct / total
    
    def fit(self, train_loader, val_loader, epochs, checkpoint_path=None):
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = self._run_epoch(train_loader, train=True)
            val_loss, val_acc = self._run_epoch(val_loader, train=False)
            if self.scheduler:
                self.scheduler.step()

            lr = self.optimizer.param_groups[0]['lr']
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)
            self.history['lr'].append(lr)

            saved = ''
            if checkpoint_path and val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.save(checkpoint_path)
                saved = '  [saved]'
            print(f"Epoch {epoch:03d} | train loss={train_loss:.4f} acc={train_acc:.4f} | "
                  f"val loss={val_loss:.4f} acc={val_acc:.4f} | lr={lr:.2e}{saved}")
        return self.history

    def evaluate(self, loader):
        self.model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(self.device)
                out = self.model(batch.x, batch.edge_index, batch.batch)
                probs = torch.softmax(out, dim=1)[:, 1]
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(batch.y.squeeze().cpu().numpy())
    
        probs, labels = np.array(all_probs), np.array(all_labels)
        
        return {
            'auc_roc': roc_auc_score(labels, probs),
            'accuracy': accuracy_score(labels, (probs > 0.5).astype(int)),
            'bg_rejection_30': _bg_rejection(labels, probs, signal_eff=0.30),
            'bg_rejection_50': _bg_rejection(labels, probs, signal_eff=0.50),
        }

    def save(self, path):
        torch.save({
            'model_state': self.model.state_dict(),
            'optimizer_state': self.optimizer.state_dict(),
            'best_val_loss': self.best_val_loss,
            'history': self.history,
        }, path)

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(ckpt['model_state'])
        self.optimizer.load_state_dict(ckpt['optimizer_state'])
        self.best_val_loss = ckpt['best_val_loss']
        self.history = ckpt['history']


## 8. Training loop

In [ ]:
histories = {}
for name, cls in MODEL_CLASSES.items():
    print(f"\n=== Training {name} ===")
    model = cls()
    lr = LR_BASE * (BATCH_SIZE / 64)               # linear scaling rule
    optimizer = Adam(model.parameters(), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    trainer = Trainer(model, optimizer, scheduler, device=DEVICE)

    checkpoint_path = os.path.join(CKPT_DIR, f'{name}_best.pt')
    trainer.fit(train_loader, val_loader, epochs=N_EPOCHS, checkpoint_path=checkpoint_path)
    histories[name] = trainer.history


## 9. Evaluation and export

In [ ]:
metrics = {}
for name, cls in MODEL_CLASSES.items():
    model = cls()
    optimizer = Adam(model.parameters())   # dummy; state is overwritten by load()
    trainer = Trainer(model, optimizer, device=DEVICE)
    trainer.load(os.path.join(CKPT_DIR, f'{name}_best.pt'))
    metrics[name] = trainer.evaluate(test_loader)
    print(f"{name}: {metrics[name]}")


In [ ]:
with open(os.path.join(CKPT_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics.json to {CKPT_DIR}")


In [ ]:
print("| Model | AUC-ROC | Accuracy | BG Rej @30% | BG Rej @50% |")
print("|---|---|---|---|---|")
for name, m in metrics.items():
    print(f"| {name} | {m['auc_roc']:.4f} | {m['accuracy']:.4f} | "
          f"{m['bg_rejection_30']:.2f} | {m['bg_rejection_50']:.2f} |")


In [ ]:
best_name = max(metrics, key=lambda n: metrics[n]['auc_roc'])
best_model = MODEL_CLASSES[best_name]()
ckpt = torch.load(os.path.join(CKPT_DIR, f'{best_name}_best.pt'), map_location=DEVICE, weights_only=False)
best_model.load_state_dict(ckpt['model_state'])

torch.save({
    'model_state': best_model.state_dict(),
    'model_name': best_name,
    'auc_roc': metrics[best_name]['auc_roc'],
    'n_params': sum(p.numel() for p in best_model.parameters()),
}, os.path.join(CKPT_DIR, 'best_model.pt'))

print(f"Best model: {best_name} (AUC-ROC={metrics[best_name]['auc_roc']:.4f})")
